In [1]:
import re
import unicodedata
import requests
import tqdm
import math

def get_text(n_phrases=200):
    # Générateur en ligne de texte français
    url = "https://enneagon.org/phrases/" + "fetch42"
    form_data = {
        "nb": str(n_phrases),
    }
    response = requests.post(url, data=form_data)
    if not response.ok:
        print("request failed")
        txt = ""
    else:
        data = response.json()
        msg = data.get("message", "")
        txt = data.get("texte", "")
    return txt

##  -------------- ENCODING SECTION --------------

ENCODE_TABLE = {key : dict(zip(range(0, 26), list(range(key, 26)) + list(range(0, key)))) for key in range(0, 26)}
DECODE_TABLE = {key : dict(zip(list(range(key, 26)) + list(range(0, key)), range(0, 26))) for key in range(0, 26)}

def normalize_encode(text, key):
    # https://www.unicode.org/reports/tr44/tr44-34.html#GC_Values_Table
    text = ''.join(c for c in unicodedata.normalize('NFD', text)
            if unicodedata.category(c) in "Lu"+"Ll").lower()
    key = ''.join(c for c in unicodedata.normalize('NFD', key)
            if unicodedata.category(c) in "Lu"+"Ll").lower()
    return text, key

def normalize_decode(text, key):
    return text.upper(), key.upper()

def encode(text, key):
    text, key = normalize_encode(text, key)
    key  = [ord(char) - 97 for char in key]
    text = [ord(char) - 97 for char in text]
    for pos, x in enumerate(text):
        alphabet = key[pos%len(key)]
        text[pos] = ENCODE_TABLE[alphabet][x]
    return str().join(map(lambda x: str(chr(x + 65)), text))

def decode(text, key):
    text, key = normalize_decode(text, key)
    key  = [ord(char) - 65 for char in key]
    text = [ord(char) - 65 for char in text]
    for pos, x in enumerate(text):
        alphabet = key[pos%len(key)]
        text[pos] = DECODE_TABLE[alphabet][x]
    return str().join(map(lambda x: chr(x + 97), text))

##  -------------- ENCRYPTING SECTION --------------

MIN_KEY_LENGTH = 5
FREQ_CHAR = "E" # langue française


def find_factors(number: int, start=MIN_KEY_LENGTH):
    if number in [0, 1]:
        return [number]
    fctrs = []
    for x in range(start, round(math.sqrt(number))+1):
        if number % x == 0:
            fctrs.append(x)
    fctrs.append(number)
    return fctrs

def babbage_kasiski(text: str, chunk_size_range=(5, 15)):
    patterns = {}
    for pos in tqdm.trange(len(text)):
        for size in range(chunk_size_range[0]-1, chunk_size_range[1]):
            if pos + size == len(text):
                break
            chunk = text[pos:pos + size + 1]
            n_occ = re.findall(f"{chunk}(.*?){chunk}", text)
            if len(n_occ) > 1:
                patterns[chunk] = n_occ
        if pos + size == len(text):
            break
    return patterns

def frequency_key(cipher, key_length, target=FREQ_CHAR):
    split_text = [cipher[i::key_length] for i in range(key_length)]
    alphabet_chars = ""
    key = ord(target.upper()) - 65
    for text in split_text:
        max_char = ord(max(text, key=text.count)) - 65
        alphabet_chars += chr((max_char - key) % 26 + 65)
    return alphabet_chars

def GCF(factors, max_num=50):
    common_factors = set(factors[0])
    for fmap in factors[1:]:
        common_factors.intersection_update(set(fmap))
    return max(common_factors, key=lambda x: x if x <= max_num else 0)



In [3]:
text = get_text()
key = "cosanostra"

cipher = encode(text, key)
print(cipher)
clear = decode(cipher, key)
print(clear)


UOURVTAXITQILAPSLMVCQBVIGWGGTEUHDEHFWMIEOCJAYONXTLCJAEVZDXRSUWKENIHTJDGZWUEGFHLVGZDEFOMILBNWUPBIJOFITOEOAGSELTUWDEAQAXLXCWEEEQWMYOOAWDRZGBUUVODIBBFXJARDDIDIWKRPCGKEHZWFVNVOMXPCFMIIDISBYSKWVLCFYEAHSOVCNOXIYZWWLNKBVUFHJBVLRCMRYOHXZNVIJERHDTJCWZHTHFWIRRCDZRNGWTDETSVEYOHTIADCDEDIWVVRVOANRGUHDEVSKDBWNXETOSLTESHELSFSUOHFSZVPQIJSBILXEITQWTGSYNVRTSDEZOJVYEFSKVNZWNISVCLOHHSKUSKJGUFONXQEVSTIRBSBDADZWDRAWERPRFWNQFWLRLWHBAVZMVVSFSJNVSJLKEODKIYGFXDEHSJOAHUARNISJDNJALKEPSRVBWDTDEURWUKCFVCEUZMNFCMYWLCBLLNTGKXEGHDAVZKXKRQINAVHETCPGBVAAHUXKTGSPPYWUTKIQBDOACMOIIVDJEPWHBKAOAWNGZSIFRVSLRRATERNVSWTTSEBJSCBLENHLTTHGNSMBBUHLEVXWMVGEHEAKFVAOOLMVMGBLSHFDXTOODLEQIUHDMKGKAVFWWVPQZACRWDRVUVWUIHBKBCEPQWDREMXCQWSKSRQGGUEUSLIYPSERNEOATFCFOVRTSTRRDSKXNGNDEOFSLHUKTMTPOKLVNGHEAAIKVIIVRWLNILXLRFIEAYAGKRLUWDARFGLKAVFWNPCFMIEWBHOVBLWRPRIAEGSDEVSCJSIGPAXECGEMEWOAFVEPQGRRAAXLXESKTVBWOZTCPDENFJBMEFOFSHBWLFLKHMDRSFMZETSWTWSEXGRKGSIFONXTRCWKOAZSFVTJCVEQSDTDEKZDEHFWZIAESVUZCFWVAOSHLNWFWIEFSNOHGFUJPKZKVRFJHETSIWTBIKGFSTOASBBFXDE

In [4]:
patterns = babbage_kasiski(cipher)
patterns

100%|████████████████████████████████████████████████████████████████████████████▊| 7191/7205 [00:09<00:00, 759.46it/s]


{'GAWNG': ['BWVVSUOARREMXCLGQZAAUWEYIUHGIESDXJBQIXFBBKWVPTCXEFGAHENGILPNGXERTVSDEHFSFFUTDJOCFWGFTTSVEICAKRNQIKIYGHKZRGBLTEOFJLINZWMRBLICAESSUASJXMETWWPECXHEDGRGNGZWGVZTIASFSDTSRUSUOARWWVQWSDQHSKALIUGAEEGAGKIOSKSRGSWZEWLLRBINXJTWEMEWSLXDMGBWAOGGKSEGDSRFSKKVPTWKEFSLVRVKRWSQSUHENCWKSNBUXJUVWDEFZWLGRGAAEEGSNIAKSFTCISOFITSFSBFLTETFITAGSSNZLRFWPNFSBKSGGVEHLHTLVTSKMNWFLJPGQLAGSMKZMRIASFOFMUEESLEGOLWVPNCJAOZWCVNGQZAAUWKRIUDSSQONBJDKGEOVDWGUAPHIURXSECAKGVOABWKJUTAGIPCEFVLGQZAFGWNIQWWDUVTSBJAKHLRRBLXJOWGGUOWWGUEOODHRIJXLXGBXAAHKWVSVWFEFOXTZRGTSCROMZRRFWWNZOJVYAPHVEFIUVVSGBKUPQWLHUKZHAEHSZVADFMYNAEXETUSDOAGSVFUVIEEVZUHDMGBUACOJXKEKBVRRZWICAHCFNVSJUCAPQWTRBFHZRDFJEGSFNGATIFCBBYKVSFSEEQSUBESRCDIGWINVSTSKEEJWYRIVSVEYOZTIDKSKSRRWFVSWWNRRXWVIOKGSVBWJLRIUWDAEOALFNROJIFWWGRYCBLFNWLNEMQINEZSFMIEHZWCUWKLVZCJSNGRWFVLCWKSRFHTJSGFFAGIJXDOSIWUFSWMSOPBWLNTXTZRGSKTQSBTWAKHVOABWEVSOCALHBSIIEUZSUGFWWRNUZWSPCFWZTKCFSQSEXIGGBUERHVXGONOJIFOLBFNFSKIAHWKVTUJSRVSFMJENCFLRGHKFVKBUEFSLEVSCILRRGYTCETSKEATAKVNVOMTNBLLLRECE

In [5]:
factors = []

for key in patterns:
    occ = patterns[key]
    for chunk in occ:
        fctrs = find_factors(len(key + chunk))
        factors.append(fctrs)

key_length = GCF(factors)
print(key_length)

10


In [6]:
key = frequency_key(cipher, key_length)
print(key)

COSANOSTRA


In [7]:
decode(cipher, key)

'sacrifiertoutacetteconditioncestleuretremoralaveclavieilleassiseaupasdeleursnouvellesaupublicpourvoiramonsalutsilencieuxaimercethommedeloidutalionnesappliquerapasseulementauxcontribuablesdelargentaveclafilledunindustrielpourlapeintureetlasculptureparaphraseameredelaparabolequecertainescometesdoiventmettreplusdecouragepoursoutenircetteguerrelemarchedesvaleurstotoutardsivousavezetebienaimabledemelapprendresalutjailucesdernierstempsilsnemeferontchangerdavistenezvoilamesdeuxoncleslunsoufflantlaforgeetlailsetrouvaitmalpendantcetteexplicationlonouvritprecipitammentlaportetremblanteetgemissanteattachezamoncouetjemismonairdabattementsurlecompteducommissairedepoliceilyeuticiunsilencedequelquessecondesetilbalancaitsonverrebrepargnezlebrasquifutcassenetmanuscritdelauteurdumalmoralsilaerostatrencontreunpointdappuietellesavaitbiencequejaimeencoremieuxcestinevitablearrivedansunesolitudeentiereetjemeprisaisavecraisonlamethodedelameilleuregracedumondeameplaindredevousnbspilsverrontquetousnosraisonnem